# Pensjonsdemografi og pensjonsvolum

Denne notebooken analyserer ferdige **Gold-data** fra Pensjon Lakehouse-pipelinen i **Databricks**.

Rapporten leser Gold-Parquet direkte fra Azure Data Lake Storage Gen2:

```text
abfss://lakehouse@pensjonlakehouse.dfs.core.windows.net/gold
```

Den leser altså ikke direkte fra Bronze- eller Silver-laget, og den kjører ikke SSB-/DuckDB-pipelinen på nytt.

Rapporten viser:

1. Utvikling i pensjonsandel 55+ over tid
2. Kommuner med høyest andel innbyggere 55+
3. Aldersgruppefordeling for nyeste år
4. Empirisk aldersfordelingskurve for nyeste år, hvis ettårsfilen finnes i Gold-laget
5. Endring i aldersgrupper fra første til siste år
6. Seniorer 55+ relativt til personer i alderen 20–54
7. Næringer med høyest estimert pensjonsvolum
8. Sammenheng mellom antall lønnstakere og månedslønn per næring

Forutsetning:

- Databricks-clusteret har tilgang til storage accounten.
- Ingen nøkler eller secrets ligger i notebooken.
- Gold-dataene er allerede skrevet til ADLS Gen2 av lakehouse-pipelinen.


## 1. Imports, Spark og ADLS-path

I Databricks finnes `spark` normalt allerede. Fallbacken under gjør at cellen også kan kjøres i en vanlig PySpark-session utenfor Databricks.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from pyspark.sql import functions as F

try:
    spark  # Databricks oppretter denne automatisk
except NameError:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("Pensjonsdemografi og pensjonsvolum")
        .getOrCreate()
    )

try:
    display
except NameError:
    from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

BASE_PATH = "abfss://lakehouse@pensjonlakehouse.dfs.core.windows.net/gold"

# Brukes bare av den valgfrie PNG-eksporten nederst.
# I Databricks er dette driverens lokale /tmp, ikke ADLS.
REPORTS = Path("/tmp/pensjon_lakehouse/reports")

print(f"Gold-path: {BASE_PATH}")


## 2. Last inn Gold-data fra ADLS Gen2

Denne cellen erstatter den lokale DuckDB-/`/tmp`-lesingen fra originalnotebooken med Spark-lesing fra `abfss://...`.

`aldersgruppe_fordeling` støtter både gammelt og nytt filnavn:

- `aldersgruppe_fordeling_siste_ar.parquet`
- `aldersgruppe_fordeling.parquet`

Ettårsfilen `aldersfordeling_siste_ar.parquet` er valgfri. Hvis den ikke finnes i Gold-laget, hopper notebooken over akkurat den empiriske alderskurven, men resten av analysen kjører videre.


In [ ]:
def read_gold_parquet(filename: str, view_name: str):
    path = f"{BASE_PATH}/{filename}"
    df = spark.read.parquet(path)
    df.createOrReplaceTempView(view_name)
    print(f"✓ {view_name}: {path}")
    return df


def read_gold_parquet_any(filenames: list[str], view_name: str):
    errors = []
    for filename in filenames:
        try:
            return read_gold_parquet(filename, view_name)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise RuntimeError(
        f"Kunne ikke lese {view_name}. Prøvde:\n" + "\n".join(errors)
    )


pensjonsandel_trend = read_gold_parquet(
    "pensjonsandel_trend.parquet",
    "pensjonsandel_trend",
)

top_kommuner = read_gold_parquet(
    "top_kommuner_pensjonsalder.parquet",
    "top_kommuner",
)

naering_pensjonsvolum = read_gold_parquet(
    "naering_pensjonsvolum.parquet",
    "naering_pensjonsvolum",
)

aldersgruppe_fordeling = read_gold_parquet_any(
    [
        "aldersgruppe_fordeling_siste_ar.parquet",
        "aldersgruppe_fordeling.parquet",
    ],
    "aldersgruppe_fordeling",
)

aldersgruppe_trend = read_gold_parquet(
    "aldersgruppe_trend.parquet",
    "aldersgruppe_trend",
)

try:
    aldersfordeling_siste_ar = read_gold_parquet(
        "aldersfordeling_siste_ar.parquet",
        "aldersfordeling_siste_ar",
    )
    has_aldersfordeling_siste_ar = True
except Exception as exc:
    aldersfordeling_siste_ar = None
    has_aldersfordeling_siste_ar = False
    print(
        "ℹ️ Valgfri fil mangler: aldersfordeling_siste_ar.parquet. "
        "Empirisk ettårsalderskurve hoppes over."
    )
    print(str(exc).split("\n")[0])


## 3. Pensjonsandel 55+ over tid

Denne tabellen og figuren viser hvordan andel innbyggere 55+ utvikler seg over tid.


In [ ]:
df_trend = pensjonsandel_trend.orderBy("year").toPandas()

# Støtter både gammel kolonne fra 01-notebooken og ny kolonne fra 02-notebooken.
if "snitt_pensjonsandel_pst" not in df_trend.columns and "pensjonsandel_pst" in df_trend.columns:
    df_trend = df_trend.rename(columns={"pensjonsandel_pst": "snitt_pensjonsandel_pst"})

df_trend["year"] = df_trend["year"].astype(int)

display(df_trend[["year", "snitt_pensjonsandel_pst", "total_55_pluss", "total_befolkning"]])


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_trend["year"],
    df_trend["snitt_pensjonsandel_pst"],
    marker="o",
)

plt.title("Pensjonsandel 55+ over tid")
plt.xlabel("År")
plt.ylabel("Pensjonsandel 55+ (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Kommuner med høyest andel 55+

Her ser vi hvilke kommuner som har størst andel innbyggere i aldersgruppen 55+.


In [ ]:
df_kommuner = (
    top_kommuner
    .select(
        "kommune_label",
        "total_befolkning",
        "pension_age_befolkning",
        F.round(F.col("pension_age_share") * 100, 1).alias("andel_55_pluss"),
    )
    .orderBy(F.col("andel_55_pluss").desc())
    .limit(10)
    .toPandas()
)

# Horisontal barplot leses enklest med lavest verdi øverst i DataFrame.
df_kommuner = df_kommuner.sort_values("andel_55_pluss", ascending=True)

display(df_kommuner)


In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    df_kommuner["kommune_label"],
    df_kommuner["andel_55_pluss"],
)

plt.title("Kommuner med høyest andel innbyggere 55+")
plt.xlabel("Andel 55+ (%)")
plt.ylabel("Kommune")
plt.tight_layout()
plt.show()


## 5. Aldersgruppefordeling nyeste år

Dette er en grov aldersfordeling basert på rapportklare aldersgrupper i Gold-laget.


In [ ]:
df_aldersfordeling = (
    aldersgruppe_fordeling
    .select(
        "aldersgruppe",
        "aldersgruppe_sortering",
        "befolkning",
        F.round(F.col("andel") * 100, 1).alias("andel_prosent"),
    )
    .orderBy("aldersgruppe_sortering")
    .toPandas()
)

display(df_aldersfordeling)


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    df_aldersfordeling["aldersgruppe"],
    df_aldersfordeling["andel_prosent"],
)

plt.title("Aldersgruppefordeling nyeste år")
plt.xlabel("Aldersgruppe")
plt.ylabel("Andel av befolkningen (%)")
plt.tight_layout()
plt.show()


## 6. Empirisk aldersfordelingskurve for nyeste år

Denne figuren bruker ettårsaldersfordelingen i Gold-laget hvis filen finnes:

```text
gold/aldersfordeling_siste_ar.parquet
```

Den svarer på spørsmålet:

> Hvordan ser befolkningens aldersprofil ut nå?


In [ ]:
if has_aldersfordeling_siste_ar:
    df_alder_siste_ar = (
        aldersfordeling_siste_ar
        .select(
            "year",
            "alder",
            "befolkning",
            (F.col("andel") * 100).alias("andel_prosent"),
        )
        .orderBy("alder")
        .toPandas()
    )

    df_alder_siste_ar["glattet_andel_prosent"] = (
        df_alder_siste_ar["andel_prosent"]
        .rolling(window=5, center=True, min_periods=1)
        .mean()
    )

    display(df_alder_siste_ar)
else:
    df_alder_siste_ar = pd.DataFrame()
    print("Hopper over: aldersfordeling_siste_ar.parquet finnes ikke i Gold-laget.")


In [ ]:
if not df_alder_siste_ar.empty:
    latest_year_age_curve = int(df_alder_siste_ar["year"].iloc[0])

    plt.figure(figsize=(11, 6))

    plt.plot(
        df_alder_siste_ar["alder"],
        df_alder_siste_ar["glattet_andel_prosent"],
        linewidth=2,
    )

    plt.fill_between(
        df_alder_siste_ar["alder"],
        df_alder_siste_ar["glattet_andel_prosent"],
        alpha=0.2,
    )

    plt.title(f"Aldersfordelingskurve for befolkningen, {latest_year_age_curve}")
    plt.xlabel("Alder")
    plt.ylabel("Andel av befolkningen (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Ingen ettårsalderskurve å plotte.")


## 7. Endring i aldersgrupper fra første til siste år

Denne figuren viser hvilke aldersgrupper som øker eller faller som andel av befolkningen.

Dette er mer informativt enn å plotte flere nesten like aldersfordelingskurver over tid.


In [ ]:
df_aldersgruppe_trend_long = (
    aldersgruppe_trend
    .select(
        "year",
        "aldersgruppe",
        "aldersgruppe_sortering",
        "befolkning",
        F.round(F.col("andel") * 100, 1).alias("andel_prosent"),
    )
    .orderBy("year", "aldersgruppe_sortering")
    .toPandas()
)

first_year = df_aldersgruppe_trend_long["year"].min()
last_year = df_aldersgruppe_trend_long["year"].max()

df_aldersgruppe_endring = (
    df_aldersgruppe_trend_long
    .pivot(
        index=["aldersgruppe", "aldersgruppe_sortering"],
        columns="year",
        values="andel_prosent",
    )
    .reset_index()
)

df_aldersgruppe_endring["endring_prosentpoeng"] = (
    df_aldersgruppe_endring[last_year] - df_aldersgruppe_endring[first_year]
)

df_aldersgruppe_endring = df_aldersgruppe_endring.sort_values("aldersgruppe_sortering")

display(df_aldersgruppe_endring)


In [ ]:
plt.figure(figsize=(10, 5))

plt.axhline(0, linewidth=1)

plt.bar(
    df_aldersgruppe_endring["aldersgruppe"],
    df_aldersgruppe_endring["endring_prosentpoeng"],
)

plt.title(f"Endring i aldersgruppenes befolkningsandel fra {first_year} til {last_year}")
plt.xlabel("Aldersgruppe")
plt.ylabel("Endring i prosentpoeng")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Seniorer relativt til yrkesaktiv alder

Denne figuren viser forholdet mellom:

- seniorer 55+
- personer i alderen 20–54

Dette gir en enkel indikator på demografisk pensjonspress.


In [ ]:
df_seniorpress = (
    aldersgruppe_trend
    .groupBy("year")
    .agg(
        F.sum(
            F.when(F.col("aldersgruppe").isin("20-34", "35-49", "50-54"), F.col("befolkning"))
            .otherwise(F.lit(0))
        ).alias("yrkesaktiv_20_54"),
        F.sum(
            F.when(F.col("aldersgruppe").isin("55-61", "62-66", "67-74", "75+"), F.col("befolkning"))
            .otherwise(F.lit(0))
        ).alias("senior_55_pluss"),
    )
    .orderBy("year")
    .toPandas()
)

df_seniorpress["senior_per_yrkesaktiv"] = (
    df_seniorpress["senior_55_pluss"]
    / df_seniorpress["yrkesaktiv_20_54"]
)

display(df_seniorpress)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_seniorpress["year"],
    df_seniorpress["senior_per_yrkesaktiv"],
    marker="o",
)

plt.title("Seniorer 55+ per person i alderen 20–54")
plt.xlabel("År")
plt.ylabel("Forholdstall")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Næringer med høyest estimert pensjonsvolum

Estimert pensjonsvolum er beregnet i pipelinen som:

```text
lønnstakere × månedslønn × 12 × 0.02
```

`Alle næringer` ekskluderes her fordi det er en totalsum, ikke en enkelt næring.


In [ ]:
df_naering = (
    naering_pensjonsvolum
    .filter(F.col("naering_label") != "Alle næringer")
    .select(
        "naering_label",
        "lonsstakere",
        "manedslonn",
        "estimert_pensjonsvolum",
        (F.col("estimert_pensjonsvolum") / F.lit(1_000_000_000)).alias("estimert_pensjonsvolum_mrd"),
    )
    .orderBy(F.col("estimert_pensjonsvolum").desc())
    .limit(10)
    .toPandas()
)

# Horisontal barplot leses enklest med lavest verdi øverst i DataFrame.
df_naering = df_naering.sort_values("estimert_pensjonsvolum_mrd", ascending=True)

display(df_naering)


In [ ]:
plt.figure(figsize=(11, 6))

plt.barh(
    df_naering["naering_label"],
    df_naering["estimert_pensjonsvolum_mrd"],
)

plt.title("Næringer med høyest estimert pensjonsvolum")
plt.xlabel("Estimert pensjonsvolum, mrd. kr")
plt.ylabel("Næring")
plt.tight_layout()
plt.show()


## 10. Lønnstakere og månedslønn per næring

Denne figuren gir et ekstra blikk på hva som driver pensjonsvolumet: mange ansatte, høy månedslønn, eller en kombinasjon.


In [ ]:
df_naering_scatter = (
    naering_pensjonsvolum
    .filter(F.col("naering_label") != "Alle næringer")
    .select(
        "naering_label",
        "lonsstakere",
        "manedslonn",
        (F.col("estimert_pensjonsvolum") / F.lit(1_000_000_000)).alias("estimert_pensjonsvolum_mrd"),
    )
    .orderBy(F.col("estimert_pensjonsvolum_mrd").desc())
    .toPandas()
)

display(df_naering_scatter)


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    df_naering_scatter["lonsstakere"],
    df_naering_scatter["manedslonn"],
)

plt.title("Lønnstakere og månedslønn per næring")
plt.xlabel("Antall lønnstakere")
plt.ylabel("Månedslønn")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Kort oppsummering

Denne cellen trekker ut noen nøkkelpunkter fra analysen.


In [ ]:
latest_year = df_trend["year"].max()

latest_share = df_trend.loc[
    df_trend["year"] == latest_year,
    "snitt_pensjonsandel_pst",
].iloc[0]

top_kommune = df_kommuner.sort_values(
    "andel_55_pluss",
    ascending=False,
).iloc[0]

top_naering = df_naering.sort_values(
    "estimert_pensjonsvolum_mrd",
    ascending=False,
).iloc[0]

largest_age_group = df_aldersfordeling.sort_values(
    "andel_prosent",
    ascending=False,
).iloc[0]

oldest_share = df_aldersfordeling.loc[
    df_aldersfordeling["aldersgruppe"] == "75+",
    "andel_prosent",
].iloc[0]

latest_seniorpress = df_seniorpress.loc[
    df_seniorpress["year"] == latest_year,
    "senior_per_yrkesaktiv",
].iloc[0]

print(f"Siste år i datasettet er {latest_year}.")
print(f"Pensjonsandel 55+ er {latest_share:.1f} %.")
print(
    f"Kommunen med høyest andel 55+ er {top_kommune['kommune_label']} "
    f"med {float(top_kommune['andel_55_pluss']):.1f} %."
)
print(
    f"Største aldersgruppe siste år er {largest_age_group['aldersgruppe']} "
    f"med {float(largest_age_group['andel_prosent']):.1f} % av befolkningen."
)

if not df_alder_siste_ar.empty:
    peak_age = df_alder_siste_ar.sort_values(
        "glattet_andel_prosent",
        ascending=False,
    ).iloc[0]
    print(
        f"Toppen i den glattede aldersfordelingen ligger rundt alder "
        f"{int(peak_age['alder'])}, med {peak_age['glattet_andel_prosent']:.2f} % av befolkningen."
    )
else:
    print("Ettårsaldersfordeling mangler, så aldersprofilens toppunkt er ikke beregnet.")

print(f"Andelen 75+ siste år er {float(oldest_share):.1f} %.")
print(f"Seniorer 55+ per person i alderen 20–54 er {latest_seniorpress:.2f}.")
print(
    f"Næringen med høyest estimert pensjonsvolum er "
    f"{top_naering['naering_label']} "
    f"med ca. {top_naering['estimert_pensjonsvolum_mrd']:.1f} mrd. kr."
)


## 12. Valgfritt: lagre grafene som PNG

Denne cellen lagrer de viktigste grafene til driverens lokale mappe:

```text
/tmp/pensjon_lakehouse/reports/
```

I Databricks er dette ikke det samme som ADLS. Bruk ADLS/DBFS-eksport hvis rapportbildene skal lagres permanent.


In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)

# Pensjonsandel over tid
plt.figure(figsize=(10, 5))
plt.plot(df_trend["year"], df_trend["snitt_pensjonsandel_pst"], marker="o")
plt.title("Pensjonsandel 55+ over tid")
plt.xlabel("År")
plt.ylabel("Pensjonsandel 55+ (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / "pensjonsandel_55_pluss_trend.png", dpi=150)
plt.close()

# Kommuner med høyest andel 55+
plt.figure(figsize=(10, 6))
plt.barh(df_kommuner["kommune_label"], df_kommuner["andel_55_pluss"])
plt.title("Kommuner med høyest andel innbyggere 55+")
plt.xlabel("Andel 55+ (%)")
plt.ylabel("Kommune")
plt.tight_layout()
plt.savefig(REPORTS / "kommuner_hoyest_andel_55_pluss.png", dpi=150)
plt.close()

# Aldersgruppefordeling nyeste år
plt.figure(figsize=(10, 5))
plt.bar(df_aldersfordeling["aldersgruppe"], df_aldersfordeling["andel_prosent"])
plt.title("Aldersgruppefordeling nyeste år")
plt.xlabel("Aldersgruppe")
plt.ylabel("Andel av befolkningen (%)")
plt.tight_layout()
plt.savefig(REPORTS / "aldersgruppefordeling_nyeste_ar.png", dpi=150)
plt.close()

# Empirisk aldersfordelingskurve nyeste år, hvis filen finnes
if not df_alder_siste_ar.empty:
    plt.figure(figsize=(11, 6))
    plt.plot(df_alder_siste_ar["alder"], df_alder_siste_ar["glattet_andel_prosent"], linewidth=2)
    plt.fill_between(df_alder_siste_ar["alder"], df_alder_siste_ar["glattet_andel_prosent"], alpha=0.2)
    plt.title(f"Aldersfordelingskurve for befolkningen, {latest_year_age_curve}")
    plt.xlabel("Alder")
    plt.ylabel("Andel av befolkningen (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORTS / "aldersfordelingskurve_nyeste_ar.png", dpi=150)
    plt.close()

# Endring i aldersgrupper
plt.figure(figsize=(10, 5))
plt.axhline(0, linewidth=1)
plt.bar(
    df_aldersgruppe_endring["aldersgruppe"],
    df_aldersgruppe_endring["endring_prosentpoeng"],
)
plt.title(f"Endring i aldersgruppenes befolkningsandel fra {first_year} til {last_year}")
plt.xlabel("Aldersgruppe")
plt.ylabel("Endring i prosentpoeng")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / "endring_aldersgrupper.png", dpi=150)
plt.close()

# Seniorer relativt til yrkesaktiv alder
plt.figure(figsize=(10, 5))
plt.plot(df_seniorpress["year"], df_seniorpress["senior_per_yrkesaktiv"], marker="o")
plt.title("Seniorer 55+ per person i alderen 20–54")
plt.xlabel("År")
plt.ylabel("Forholdstall")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / "seniorer_per_yrkesaktiv.png", dpi=150)
plt.close()

# Næringer med høyest estimert pensjonsvolum
plt.figure(figsize=(11, 6))
plt.barh(df_naering["naering_label"], df_naering["estimert_pensjonsvolum_mrd"])
plt.title("Næringer med høyest estimert pensjonsvolum")
plt.xlabel("Estimert pensjonsvolum, mrd. kr")
plt.ylabel("Næring")
plt.tight_layout()
plt.savefig(REPORTS / "naeringer_hoyest_estimert_pensjonsvolum.png", dpi=150)
plt.close()

print(f"Grafer lagret i: {REPORTS}")
